In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
# ==========================================
# SYSTEM INSTALL (run once)
# ==========================================
!sudo apt-get update -y
!sudo apt-get install -y espeak-ng libsndfile1-dev ffmpeg

# ==========================================
# PYTHON DEPENDENCIES (clean & compatible)
# ==========================================
import sys

!{sys.executable} -m pip install --upgrade pip

!{sys.executable} -m pip install \
torch==2.4.0 torchaudio==2.4.0 \
transformers==4.44.2 \
coqui-tts \
soundfile librosa pandas huggingface_hub

# ==========================================
# REMOVE CONFLICTING PACKAGES
# ==========================================
!{sys.executable} -m pip uninstall -y torchvision cupy-cuda12x

Hit:1 https://download.docker.com/linux/debian bullseye InRelease
Hit:2 https://deb.debian.org/debian bullseye InRelease                         
Hit:3 https://deb.debian.org/debian-security bullseye-security InRelease       
Hit:4 https://deb.debian.org/debian bullseye-updates InRelease            
Get:5 https://nvidia.github.io/libnvidia-container/stable/deb/amd64  InRelease [1477 B]
Hit:6 https://packages.cloud.google.com/apt gcsfuse-bullseye InRelease
Hit:7 https://packages.cloud.google.com/apt google-compute-engine-bullseye-stable InRelease
Hit:8 https://packages.cloud.google.com/apt cloud-sdk-bullseye InRelease
Hit:9 https://packages.cloud.google.com/apt google-fast-socket InRelease
Fetched 1477 B in 1s (1414 B/s)
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak-ng is already the newest version (1.50+dfsg-7+deb11u2).
ffmpeg is already the newest version (7:4.3.9-0+deb11u2).
libsndfile1-dev is alr

In [ ]:
import torch
from TTS.api import TTS

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("TTS import successful ✅")

Torch: 2.4.0+cu121
CUDA: True
GPU: Tesla T4
TTS import successful ✅


In [ ]:
# ==========================================
# 0. IMPORTS & CONFIG
# ==========================================
import os
import re
import unicodedata
import glob
import pandas as pd
import soundfile as sf
import librosa
import torch
from huggingface_hub import snapshot_download, hf_hub_download


from TTS.tts.configs.shared_configs import BaseAudioConfig, BaseDatasetConfig, CharactersConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.utils.audio import AudioProcessor
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.vits import Vits, VitsCharacters
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from trainer import Trainer, TrainerArgs
from TTS.api import TTS

HF_TOKEN       = userdata.get("HF_TOKEN")
REPO_ID        = "ankitdhiman/haryanvi-tts"
TARGET_SR      = 22050
DATA_DIR       = "haryanvi_vits_data"
WAV_DIR        = os.path.join(DATA_DIR, "wavs")
LOCAL_REPO     = "hf_dataset_cache"
PHONEME_CACHE  = os.path.join(DATA_DIR, "phoneme_cache")
# PRETRAINED_DIR = "ai4bharat_hindi_vits"   # where Hindi checkpoint lands
PRETRAINED_DIR = "syspin_hindi_vits"
TEXT_MIN_WORDS = 3
TEXT_MAX_WORDS = 25

os.makedirs(WAV_DIR,        exist_ok=True)
os.makedirs(PHONEME_CACHE,  exist_ok=True)
os.makedirs(PRETRAINED_DIR, exist_ok=True)

In [ ]:
# ==========================================
# 1. DOWNLOAD HARYANVI DATASET
# ==========================================
_marker = os.path.join(LOCAL_REPO, "metadata.csv")

if os.path.exists(_marker):
    print(f"✅ Dataset already at '{LOCAL_REPO}' — skipping.")
    local_repo = LOCAL_REPO
else:
    print("--- Downloading Haryanvi dataset from HuggingFace ---")
    local_repo = snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir=LOCAL_REPO,
    )
    print(f"  Dataset downloaded to: {local_repo}")

# ==========================================
# HARDCODED TRANSLITERATION MAPS
# Based on exact characters found in your dataset
# ==========================================

# Bengali → Devanagari (character-level)
# অহঙ্কারী = अहंकारी (arrogance/pride)
BENGALI_DEVANAGARI_MAP = {
    # Vowels
    'অ': 'अ',  'আ': 'आ', 'ই': 'इ', 'ী': 'ी',
    'উ': 'उ',  'এ': 'ए', 'ও': 'ओ',
    # Consonants
    'ক': 'क',  'খ': 'ख', 'গ': 'ग', 'ঘ': 'घ',
    'ঙ': 'ङ',  'চ': 'च', 'ছ': 'छ', 'জ': 'ज',
    'ঝ': 'झ',  'ট': 'ट', 'ঠ': 'ठ', 'ড': 'ड',
    'ঢ': 'ढ',  'ণ': 'ण', 'ত': 'त', 'থ': 'थ',
    'দ': 'द',  'ধ': 'ध', 'ন': 'न', 'প': 'प',
    'ফ': 'फ',  'ব': 'ब', 'ভ': 'भ', 'ম': 'म',
    'য': 'य',  'র': 'र', 'ল': 'ल', 'শ': 'श',
    'ষ': 'ष',  'স': 'स', 'হ': 'ह',
    # Vowel signs (matras)
    'া': 'ा',  'ি': 'ि', 'ী': 'ी', 'ু': 'ु',
    'ূ': 'ू',  'ে': 'े', 'ৈ': 'ै', 'ো': 'ो',
    'ৌ': 'ौ',
    # Halant / virama
    '্': '्',
    # Anusvara / visarga
    'ং': 'ं',  'ঃ': 'ः', 'ঁ': 'ँ',
}

# Arabic/Urdu → Devanagari
# موج = मौज (mauj = joy/wave)
URDU_DEVANAGARI_MAP = {
    'ا': 'अ',  'آ': 'आ', 'ب': 'ब', 'پ': 'प',
    'ت': 'त',  'ٹ': 'ट', 'ث': 'स', 'ج': 'ज',
    'چ': 'च',  'ح': 'ह', 'خ': 'ख', 'د': 'द',
    'ڈ': 'ड',  'ذ': 'ज', 'ر': 'र', 'ڑ': 'ड़',
    'ز': 'ज़', 'ژ': 'ज', 'س': 'स', 'ش': 'श',
    'ص': 'स',  'ض': 'ज़','ط': 'त', 'ظ': 'ज़',
    'ع': 'अ',  'غ': 'ग़','ف': 'फ', 'ق': 'क',
    'ک': 'क',  'گ': 'ग', 'ل': 'ल', 'م': 'म',
    'ن': 'न',  'ں': 'ं', 'و': 'व', 'ہ': 'ह',
    'ھ': 'ह',  'ء': '',  'ی': 'य', 'ے': 'य',
    'ئ': 'य',
    # Urdu diacritics → drop
    'َ': '', 'ِ': '', 'ُ': '', 'ّ': '',
    'ْ': '', 'ً': '', 'ٍ': '', 'ٌ': '',
}

ARABIC_RE  = re.compile(r'[\u0600-\u06FF\u0750-\u077F]+')
BENGALI_RE = re.compile(r'[\u0980-\u09FF]+')


def transliterate_urdu_segment(text: str) -> str:
    return ''.join(URDU_DEVANAGARI_MAP.get(c, c) for c in text)


def transliterate_bengali_segment(text: str) -> str:
    return ''.join(BENGALI_DEVANAGARI_MAP.get(c, c) for c in text)


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = BENGALI_RE.sub(lambda m: transliterate_bengali_segment(m.group(0)), text)
    text = ARABIC_RE.sub(lambda m: transliterate_urdu_segment(m.group(0)), text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r" +", " ", text).strip()
    return text


✅ Dataset already at 'hf_dataset_cache' — skipping.


In [ ]:
# ==========================================
# 2: READ & PREPROCESS METADATA
# ==========================================
print("--- Reading and preprocessing metadata ---")

df_meta = pd.read_csv(os.path.join(local_repo, "metadata.csv"))
df_meta = df_meta.dropna(subset=["text"])
df_meta = df_meta[df_meta["text"].str.strip().str.lower() != "nan"]
df_meta["text"] = df_meta["text"].str.replace('\n', ' ')
df_meta = df_meta.drop_duplicates(subset="text")



# Apply full normalization (transliterates Bengali + Arabic in-place)
df_meta["text"] = df_meta["text"].apply(lambda x: normalize_text(str(x)))

df_meta["text"] = df_meta["text"].str.replace(r'[\n\r]+', ' ', regex=True)

# Strip leading/trailing spaces and newlines
df_meta["text"] = df_meta["text"].str.strip()

# Word-count filter (after normalization so word count is accurate)
df_meta = df_meta[
    df_meta["text"].apply(
        lambda x: TEXT_MIN_WORDS <= len(x.split()) <= TEXT_MAX_WORDS
    )
].reset_index(drop=True)


print(f"  Final shape: {df_meta.shape}")

--- Reading and preprocessing metadata ---
  Final shape: (2764, 2)


In [ ]:
_out_meta = os.path.join(DATA_DIR, "metadata.csv")


# ==========================================
# 3: FORCE CACHE INVALIDATION
# Delete old metadata.csv so normalization reruns cleanly
# ==========================================
import shutil

if os.path.exists(_out_meta):
    os.remove(_out_meta)
    print("🗑️  Deleted old metadata.csv — will reprocess with updated normalization.")

# Also clear phoneme cache since text has changed
if os.path.exists(PHONEME_CACHE):
    shutil.rmtree(PHONEME_CACHE)
    os.makedirs(PHONEME_CACHE, exist_ok=True)
    print("🗑️  Cleared phoneme cache.")


if os.path.exists(_out_meta):
    print(f"✅ Processed wavs exist at '{DATA_DIR}' — skipping.")
else:
    print("--- Processing audio files ---")
    metadata_ljspeech = []
    skipped = 0

    for idx, row in df_meta.iterrows():
        rel_path   = str(row["file_name"]).strip()
        text_val = normalize_text(str(row["text"]).strip())
        audio_path = os.path.join(local_repo, rel_path)

        if not os.path.exists(audio_path):
            skipped += 1
            continue
        try:
            audio_array, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)
        except Exception as e:
            print(f"  [{idx}] SKIP — {e}")
            skipped += 1
            continue

        fname = f"audio_{idx}"
        sf.write(os.path.join(WAV_DIR, f"{fname}.wav"), audio_array, TARGET_SR)
        # LJSpeech format: id|text|text  (normalised text = raw text for Devanagari)
        metadata_ljspeech.append(f"{fname}|{text_val}|{text_val}")

    assert len(metadata_ljspeech) > 0, f"❌ No samples exported! {skipped} skipped."

    with open(_out_meta, "w", encoding="utf-8") as f:
        f.write("\n".join(metadata_ljspeech))

    print(f"✅ Exported {len(metadata_ljspeech)} samples | Skipped: {skipped}")

🗑️  Cleared phoneme cache.
--- Processing audio files ---
✅ Exported 2764 samples | Skipped: 0


In [ ]:
# ==========================================
# 4. DOWNLOAD SYSPIN COQUI-NATIVE HINDI VITS
# ==========================================
HINDI_VITS_REPO  = "SYSPIN/vits_Hindi_Female"
PRETRAINED_MODEL = os.path.join(PRETRAINED_DIR, "best_model.pth")
PRETRAINED_CFG   = os.path.join(PRETRAINED_DIR, "config.json")

os.makedirs(PRETRAINED_DIR, exist_ok=True)

if os.path.exists(PRETRAINED_MODEL) and os.path.exists(PRETRAINED_CFG):
    print(f"✅ SYSPIN Hindi checkpoint already at '{PRETRAINED_DIR}' — skipping.")
else:
    print(f"--- Downloading Coqui-native Hindi VITS from '{HINDI_VITS_REPO}' ---")
    for fname in ["best_model.pth", "config.json"]:
        hf_hub_download(
            repo_id=HINDI_VITS_REPO,
            filename=fname,
            token=HF_TOKEN,
            local_dir=PRETRAINED_DIR,
        )
    print(f"✅ Downloaded to '{PRETRAINED_DIR}'")



✅ SYSPIN Hindi checkpoint already at 'syspin_hindi_vits' — skipping.


In [ ]:
# ==========================================
# 5. DEVANAGARI CHARACTER SET  (unchanged)
# ==========================================

DEVANAGARI_CHARS = (
    # Vowels + candra forms
    "अआइईउऊऋऌएऐओऔऑऒ"
    # Anusvara, visarga, chandrabindu
    "ँंः"
    # Consonants
    "कखगघङचछजझञटठडढणतथदधनपफबभमयरलळवशषसह"
    # Nukta consonants (loanwords)
    "क़ख़ग़ज़ड़ढ़फ़य़"
    # Vowel signs + halant + misc
    "ािीुूृॄॅॆेैॉॊोौ्ॐ"
    # Rare vowels
    "ॠॡ"
    # Devanagari digits
    "०१२३४५६७८९"
    # ALL ASCII digits (0-9)
    "0123456789"
    # ALL uppercase Latin
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    # ALL lowercase Latin
    "abcdefghijklmnopqrstuvwxyz"
)

characters_config = CharactersConfig(
    characters_class="TTS.tts.models.vits.VitsCharacters",
    pad="<PAD>",
    eos="<EOS>",
    bos="<BOS>",
    blank="<BLNK>",
    characters=DEVANAGARI_CHARS,
    punctuations="!,-.?।॥ '\"():",
    phonemes=None,
)




In [ ]:
# ==========================================
# 6. VITS CONFIG
# ==========================================
dataset_config = BaseDatasetConfig(
    formatter="ljspeech",
    meta_file_train="metadata.csv",
    path=DATA_DIR,
)
audio_config = BaseAudioConfig(
    sample_rate=TARGET_SR,
    win_length=1024,
    hop_length=256,
    fft_size=1024,
    num_mels=80,
    mel_fmin=0.0,
    mel_fmax=8000,
    preemphasis=0.0,
    ref_level_db=20,
    min_level_db=-100,
    do_trim_silence=True,
    trim_db=40,
)
config = VitsConfig(
    audio=audio_config,
    run_name="vits_haryanvi_ai4bharat",
    lr=1e-5,
    lr_disc=1e-5,
    batch_size=16,
    eval_batch_size=8,
    num_loader_workers=4,
    num_eval_loader_workers=4,
    mixed_precision=True,
    run_eval=True,
    test_delay_epochs=20,
    run_eval_steps=1680,
    epochs=300,
    save_step=334,
    save_best_after=2000,
    use_phonemes=False,
    characters=characters_config,
    compute_input_seq_cache=True,
    phoneme_cache_path=PHONEME_CACHE,
    print_step=500,
    datasets=[dataset_config],
)


In [ ]:

# ==========================================
# 7. TRAINING
# ==========================================

print("\n--- Initializing VITS model ---")

ap = AudioProcessor.init_from_config(config)

train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_max_size=50,
    eval_split_size=0.1,
)
print(f"  Train samples : {len(train_samples)}")
print(f"  Eval  samples : {len(eval_samples)}")

tokenizer, config = TTSTokenizer.init_from_config(config)
model = Vits(config, ap, tokenizer, speaker_manager=None)


# ─── Early Stopping ───────────────────────────────────────────────────────────
BEST_LOSS        = None
PATIENCE         = 0
MAX_PATIENCE     = 10             # stop after 10 consecutive evals with no improvement
EVAL_EVERY_N_EPOCHS = 10          # only act on every 10th epoch inside the callback

def early_stopping_fn(trainer):
    global BEST_LOSS, PATIENCE

    # callback fires every eval, but logic only runs every 10 epochs
    if trainer.epochs_done % EVAL_EVERY_N_EPOCHS != 0:
        return False

    current_loss = trainer.keep_avg_eval["avg_loss"]

    if BEST_LOSS is None:
        BEST_LOSS = current_loss
        return False

    if current_loss < BEST_LOSS:
        BEST_LOSS = current_loss
        PATIENCE  = 0
        print(f"  ✅ Eval loss improved → {BEST_LOSS:.4f}  (patience reset)")
        return False
    else:
        PATIENCE += 1
        print(f"  ⏳ No improvement. Patience {PATIENCE}/{MAX_PATIENCE}  (best={BEST_LOSS:.4f})")
        if PATIENCE >= MAX_PATIENCE:
            print("  🛑 Early stopping triggered!")
            return True
        return False
# ──────────────────────────────────────────────────────────────────────────────


# ─── Checkpoint Strategy ──────────────────────────────────────────────────────
LAST_RUN_DIR = None
if os.path.exists("tts_output"):
    prev_runs = sorted(
        [os.path.join("tts_output", d)
         for d in os.listdir("tts_output")
         if config.run_name in d and os.path.isdir(os.path.join("tts_output", d))],
        key=os.path.getmtime,
    )
    if prev_runs:
        LAST_RUN_DIR = prev_runs[-1]


if LAST_RUN_DIR:
    # ── RESUME from previous run (restores step counter + optimizer state) ──
    print(f"\n--- Resuming from previous run ---")
    print(f"  Directory : {LAST_RUN_DIR}")

    trainer_args = TrainerArgs(
        continue_path=LAST_RUN_DIR,   # full resume — same dir, same step counter

    )

else:
    # ── FIRST RUN: load SYSPIN Hindi pretrained weights ──
    print(f"\n--- No previous run found. you will have to train from starting---")

    print(f"\n--- uncomment the below code to to do so")

    # print(f"\n--- No previous run found. Loading SYSPIN Hindi weights ---")

    # ckpt        = torch.load(PRETRAINED_MODEL, map_location="cpu")
    # src_weights = ckpt.get("model", ckpt)
    # model_state = model.state_dict()

    # filtered_weights = {
    #     k: v for k, v in src_weights.items()
    #     if k in model_state and v.shape == model_state[k].shape
    # }
    # skipped = {
    #     k: (list(v.shape), list(model_state[k].shape))
    #     for k, v in src_weights.items()
    #     if k in model_state and v.shape != model_state[k].shape
    # }
    # if skipped:
    #     print(f"\n  {len(skipped)} size-mismatched key(s) (vocab size change — expected):")
    #     for k, (s, d) in skipped.items():
    #         print(f"      {k}: checkpoint{s} → model{d}")

    # result = model.load_state_dict(filtered_weights, strict=False)
    # print(f"\n  Missing   : {len(result.missing_keys)}")
    # print(f"  Unexpected: {len(result.unexpected_keys)}")

    # core_keywords = {
    #     "text_encoder":       "text encoder",
    #     "waveform_decoder":   "decoder (HiFiGAN)",
    #     "flow":               "normalizing flow",
    #     "duration_predictor": "duration predictor",
    #     "posterior_encoder":  "posterior encoder",
    # }
    # print("\n  === CORE LAYER AUDIT ===")
    # loaded_keys = set(filtered_weights.keys())
    # for prefix, label in core_keywords.items():
    #     loaded  = [k for k in loaded_keys        if k.startswith(prefix)]
    #     missing = [k for k in result.missing_keys if k.startswith(prefix)]
    #     status  = "✅ 100%" if loaded and not missing else "PARTIAL" if loaded else "❌ FAILED"
    #     print(f"    [{prefix}] loaded={len(loaded):>3}  missing={len(missing):>3}  → {status}  ({label})")

    # # Freeze text encoder — keeps Hindi phonetics, only voice/prosody adapts
    # frozen_count = 0
    # for name, param in model.named_parameters():
    #     if "text_encoder" in name:
    #         param.requires_grad = False
    #         frozen_count += 1
    # print(f"\n  🔒 Frozen {frozen_count} text_encoder parameters")

    # trainer_args = TrainerArgs(
    #     grad_clip=1.0,
    #     scheduler_after_epoch=False,
    # )
# ──────────────────────────────────────────────────────────────────────────────


trainer = Trainer(
    trainer_args,
    config,
    output_path="tts_output",
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

print("\n--- Starting Training ---")
trainer.fit()

In [ ]:
# ==========================================
# 8. INFERENCE
# ==========================================
print("\n--- Running Inference ---")

output_dirs = sorted(
    [os.path.join("tts_output", d)
     for d in os.listdir("tts_output") if config.run_name in d],
    key=os.path.getmtime,
)
latest_run_dir = output_dirs[-1]
config_path    = os.path.join(latest_run_dir, "config.json")

pth_files = glob.glob(os.path.join(latest_run_dir, "*.pth"))
assert len(pth_files) > 0, f"❌ No .pth checkpoint in: {latest_run_dir}"

best = [f for f in pth_files if "best_model" in os.path.basename(f)]
checkpoint_path = best[0] if best else sorted(pth_files, key=os.path.getmtime)[-1]
print(f"  Using checkpoint: {os.path.basename(checkpoint_path)}")

tts = TTS(
    model_path=checkpoint_path,
    config_path=config_path,
    progress_bar=False,
    gpu=torch.cuda.is_available(),
)

# ✅ CHANGED: Long, complex Haryanvi sentence — tests prosody, pauses, numbers, conjunctions
test_sentence = (
    "म्हारे गाम म्ह आज सुबह तैं ही बड़ी जोरदार बारिश होरी सै, "
    "अर खेतां म्ह पाणी भर गया सै, "
    "किसान भाई बड़े चिंतित सैं अर सरकार तैं मदद मांग रहे सैं।"
)

output_file = "test_haryanvi_output.wav"
tts.tts_to_file(text=test_sentence, file_path=output_file)
print(f"✅ Done! Audio saved to: {output_file}")


--- Running Inference ---
  Using checkpoint: best_model_16731.pth


/opt/conda/envs/pytorch/lib/python3.10/site-packages/TTS/api.py:92: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


✅ Done! Audio saved to: test_haryanvi_output.wav


In [ ]:
# Haryanvi test sentence
test_sentence = "जिब हाथी जंगल मं अपणा पेट खाली करै सै तो उसका गोबर कौन चट कर ज्या सै|"
output_file   = "test_haryanvi_output2.wav"
tts.tts_to_file(text=test_sentence, file_path=output_file)

print(f"✅ Done! Audio saved to: {output_file}")

जिब हाथी जंगल मं अपणा पेट खाली करै सै तो उसका गोबर कौन चट कर ज्या सै|
Character '|' not found in the vocabulary. Discarding it.


✅ Done! Audio saved to: test_haryanvi_output2.wav


In [ ]:
test_sentence ='''रै, यो सब कौण चट कर गा! जंगल मं कुछ भी बेकार नी जांदा|
किम्मे भी नही|
जिब तेंदुए अपणा खाणा ख़त्म नी करदे,
तो बचा होड़ा खाणा कौण खावै सैं'''

output_file   = "test_haryanvi_output3.wav"
tts.tts_to_file(text=test_sentence, file_path=output_file)

print(f"✅ Done! Audio saved to: {output_file}")


जिब तेंदुए अपणा खाणा ख़त्म नी करदे,
Character 'ख़' not found in the vocabulary. Discarding it.
तो बचा होड़ा खाणा कौण खावै सैं
Character 'ड़' not found in the vocabulary. Discarding it.


✅ Done! Audio saved to: test_haryanvi_output3.wav


In [ ]:
# List of Bangru/Haryanvi test sentences
test_sentences = [
    "रै, काल रात नै इतणी तेज आँधी आई कि सारा गाँव धूल तै भर ग्या अर किम्मे भी साफ नी दिख रया था|",
    "अर सुन, जिब खेतां मं पानी भर जावै सै ना, तै बैल भी चालण तै कतरावै सै अर किसान परेशान हो जावै सै|",
    "यो छोरा सारा दिन मोबाइल मं घुस्या रहवै सै, ना घर का काम देखै सै अर ना पढ़ाई की कोई फिकर करै सै|",
    "जिब तै बरसात शुरू होई सै, तब तै गलियां मं कीचड़ हो रया सै अर आवण-जावण मं भी बड़ी दिक्कत आवै सै|",
    "रै, या दुनिया भी बड़ी अजीब सै, जिब जरूरत हो सै तब कोई साथ नी देवे, अर जिब सब ठीक हो जावै सै तै सब नेड़े आ जावै सै|",
    "घर मं जिब बिजली चली जावै सै ना, तै सबने पंखा याद आवै सै अर कोई भी चैन तै बैठ नी सकै|",
    "यो कुत्ता भी बड़ा चालाक सै, जिब देखै सै कोई ना सै तै चुपके तै रोटी उठा के भाग जावै सै|",
    "जिब बूढ़े लोग आपणी पुराणी बातें सुनावै सै ना, तै लागै सै जैसे टाइम पीछे चल ग्या हो अर सब कुछ आंखां के आगे हो रया हो|"
]

# Loop through sentences and generate audio files
for i, sentence in enumerate(test_sentences, start=4):
    output_file = f"test_haryanvi_output{i}.wav"

    tts.tts_to_file(text=sentence, file_path=output_file)

    print(f"✅ Done! Audio saved to: {output_file}")

✅ Done! Audio saved to: test_haryanvi_output4.wav
✅ Done! Audio saved to: test_haryanvi_output5.wav
✅ Done! Audio saved to: test_haryanvi_output6.wav
✅ Done! Audio saved to: test_haryanvi_output7.wav
✅ Done! Audio saved to: test_haryanvi_output8.wav
✅ Done! Audio saved to: test_haryanvi_output9.wav
✅ Done! Audio saved to: test_haryanvi_output10.wav
✅ Done! Audio saved to: test_haryanvi_output11.wav


In [ ]:
test_sentence ='''एक बेर की बात सै, एक छोटा सा गाम होया करै। उस गाम मं रामफल नाम का छोरा रहै सै। रामफल बड़ा जिगरी अर मेहनती था, पर किस्मत उसकै साथ जादा ना दे सै।

एक दिन वो जंगल मं लकड़ी काटण ग्या। उड़े उसनै एक बूढ़ा आदमी दिख्या, जो घणा थक्ग्या सा लागै था। रामफल नै बिना सोचे उसनै पानी पिलाया अर आराम करन दिया।

बूढ़ा आदमी खुश होग्या अर बोल्या, "बेटा, तू घणा नेक सै। यो ले, ये बीज ले जा, अर अपने खेत मं बो दिये।"

रामफल नै बीज बो दिये। कुछ दिनां मं उड़े सोने जई फसल उग्गी। पूरा गाम देख कै हैरान रह्ग्या।

रामफल नै कदे घमंड ना किया, अर सब गाम वाल्यां के साथ बांट लिया।

तभी तै गाम मं सब कहण लागे—भलाई करैगा, तो फल जरूर मिलैगा।'''

output_file   = "test_haryanvi_outputfinal.wav"
tts.tts_to_file(text=test_sentence, file_path=output_file)

print(f"✅ Done! Audio saved to: {output_file}")

तभी तै गाम मं सब कहण लागे—भलाई करैगा, तो फल जरूर मिलैगा।
Character '—' not found in the vocabulary. Discarding it.


✅ Done! Audio saved to: test_haryanvi_outputfinal.wav


In [ ]:
test_sentence ='''एक बेर की बात सै, हरियाणे के एक छोटे से गाम मं सुरता नाम का छोरा रहै सै। सुरता दिल का घणा साफ अर मेहनती था, पर घर की हालत कुछ खास ना थी। बापू बूढ़े हो लिये थे अर खेत भी ज्यादा ना थे।

सुरता रोज सवेरे उठ कै खेतां मं काम करै, फेर जंगल मं लकड़ी काटण जावै। एक दिन वो जंगल मं घुस्स्या तो उसनै एक अजीब सी आवाज सुनाई दी। वो डर्या भी, पर हिम्मत करकै आगे बढ़्या।

आगे देख्या तो एक घायल हिरन पड़ा था। उसकै पांव मं कांटा घुस्या होया था। सुरता नै धीरे-धीरे उसनै पकड़्या अर कांटा निकाल दिया। हिरन दर्द मं तड़प रह्या था, पर थोड़ी देर बाद आराम मिल्ग्या।

सुरता बोल्या, "जा भाई, तू भी अपनी जान बचा ले।" हिरन कुछ देर उसनै देखता रह्या, जैसे शुक्रिया कर रह्या हो, फेर जंगल मं गायब होग्या।

कुछ दिन बाद सुरता फेर उसी जंगल मं ग्या। इस बार उसनै एक पुराणा पेड़ के नीचे चमकती चीज दिखी। जाके देख्या तो एक छोटा सा डिब्बा था। खोल्या तो उसमं चांदी के सिक्के अर एक पर्ची पड़ी थी।

पर्ची मं लिख्या था, "जो दिल तै मदद करै सै, किस्मत भी उसी का साथ दे सै।"

सुरता समझ ग्या के यो उसी भलाई का फल सै। पर उसनै सारे पैसे अपने पास ना रखे। गाम मं जितने भी गरीब थे, सबकी मदद कर दी। किसे के घर अनाज भिजवा दिया, किसे के बच्चे की पढ़ाई का खर्चा उठा लिया।

धीरे-धीरे सुरता का नाम पूरे इलाके मं फैल ग्या। लोग उसनै इज्जत देण लागे। पर सुरता मं घमंड का नाम तक ना था। वो आज भी पहले की तरह सादा जीवन जीया करै।

एक दिन गाम मं सूखा पड़ ग्या। पानी की भारी किल्लत हो ली। सारे गाम वाले परेशान होगए। सुरता नै सोचा के कुछ करना पड़ेगा।

वो पुराने कुएं के पास ग्या, जो कई सालां तै सूखा पड़ा था। सुरता नै गाम वाल्यां नै इकट्ठा किया अर बोल्या, "अगर सारे मिलकै मेहनत करांगे, तो यो कुआं फेर तै चालू हो सके सै।"

शुरू मं कई लोग हिचकिचाए, पर फेर सब मान गए। दिन-रात मेहनत चालू हो ली। मिट्टी हटाई गई, पत्थर निकाले गए।

तीन दिन बाद अचानक पानी की हल्की धार निकली। सारे खुश होगए। कुछ और मेहनत के बाद कुआं पूरा भर ग्या।

गाम मं फेर तै खुशहाली आ गी। लोग सुरता की तारीफ करन लागे, पर सुरता बस मुस्कुरा कै बोल्या, "ये सब आपस की एकता का फल सै।"

कुछ टाइम बाद वो ही हिरन फेर तै सुरता के खेत के पास दिख्या। इस बार वो बिलकुल ठीक था। वो सुरता के चारों तरफ घूम्या अर शांत खड़ा रह्ग्या, जैसे आशीर्वाद दे रह्या हो।

सुरता नै आसमान की तरफ देख्या अर मन ही मन धन्यवाद किया।

तभी तै गाम मं एक बात मशहूर हो गी—"जितनी सच्ची नीयत, उतनी बड़ी बरकत।"

अर यो कहानी हमें सिखावै सै के भलाई का फल जरूर मिलै सै, चाहे देर तै ही क्यों ना मिलै।'''

output_file   = "test_haryanvi_output5min.wav"
tts.tts_to_file(text=test_sentence, file_path=output_file)

print(f"✅ Done! Audio saved to: {output_file}")

✅ Done! Audio saved to: test_haryanvi_output5min.wav


In [ ]:
test_sentence ='''एक गाम मं रघुबीर नाम का आदमी रहै सै, जो थोड़ा सा जुगाड़ू अर थोड़ा सा चालाक किस्म का था। गाम वाले उसनै जानते थे, पर कोई खास भरोसा ना करै।

रघुबीर का एक छोरा था—मुन्ना। मुन्ना बिलकुल उल्टा था, दिल का साफ अर सीधा-साधा। एक दिन मुन्ना स्कूल तै आवै था, तो रास्ते मं उसनै एक बटुआ मिल्या।

मुन्ना घर आ कै बोल्या, "बापू, देखो मन्नै के मिल्या!"
रघुबीर नै बटुआ खोल्या—उसमें पैसे अर कागज थे।

रघुबीर की आंख चमक गई। बोल्या, "ए मुन्ना, यो अपने पास राख लेंगे, किसे ने के पता लगेगा।"
मुन्ना घबरा ग्या, "ना बापू, यो ठीक ना सै। जिसकू खोया होगा, वो परेशान हो रह्या होगा।"

रघुबीर थोड़ा चुप हो ग्या, फेर बोल्या, "अच्छा ठीक सै, देख लेंगें किसका सै।"

अगले दिन गाम मं ऐलान होया के किसी का बटुआ खो ग्या सै। एक बूढ़ा आदमी, नाम था हरिया, वो ढूंढता फिर रह्या था।

मुन्ना नै रघुबीर तै कहा, "बापू, चलो दे आवां।"
रघुबीर पहले तो टालता रह्या, पर मुन्ना की जिद के आगे मान ग्या।

दोन्ने हरिया के पास गए। मुन्ना बोल्या, "चाचा, यो आपका बटुआ सै के?"
हरिया की आंखां मं आंसू आ गए। "हाँ बेटा, यो ही सै! मैं तो समझा था ग्या काम तै।"

हरिया नै धन्यवाद किया अर इनाम देण की बात करी।
रघुबीर तुरन्त बोल्या, "ना-ना, इसकी जरूरत ना सै।"
पर मुन्ना बोल्या, "चाचा, आशीर्वाद दे दो बस।"

हरिया खुश होग्या अर बोला, "बेटा, तू सच्चा आदमी बनेगा।"

कुछ दिन बाद रघुबीर के खेत मं अचानक आग लग गी। हवा तेज थी, आग फैलण लागी। रघुबीर घबरा ग्या।

पूरा गाम आग बुझाण मं लग ग्या, अर सबसे आगे हरिया था। वो अपने घर तै पानी के घड़े लाया अर मदद करी।

काफी मेहनत के बाद आग बुझ गी। रघुबीर का नुकसान तो होया, पर पूरा खेत बच ग्या।

रघुबीर चुप खड़ा रह्या। उसनै समझ आ गी के आज जो गाम वाले मदद कर रहे सैं, वो सब उसी भलाई का फल सै।

रघुबीर मुन्ना के पास आया अर बोल्या, "बेटा, आज तू मन्नै सच्चाई सिखा दी।"

मुन्ना मुस्कुरा दिया, "बापू, सच्चाई मं ही ताकत सै।"

उस दिन तै रघुबीर बदल ग्या। अब वो किसी का हक ना मारता, अर गाम वाल्यां की मदद करै।

धीरे-धीरे गाम मं उसकी इज्जत भी बढ़ गी।

गाम मं लोग कहण लागे—"जुगाड़ तै काम चल जावे सै, पर इज्जत सिर्फ सच्चाई तै मिलै सै।"

अर यो बात सच्ची सै—इंसान की पहचान उसके कामां तै होवै सै, ना के उसकी चालाकी तै।'''

output_file   = "test_haryanvi_output5mintoughtest.wav"
tts.tts_to_file(text=test_sentence, file_path=output_file)

print(f"✅ Done! Audio saved to: {output_file}")

✅ Done! Audio saved to: test_haryanvi_output5mintoughtest.wav
